# analysis_tools

> MCP tools for analyzing notebook dependencies and import structure

In [ ]:
#| default_exp tools.analysis

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Set, Tuple

import os
import re
import ast
import textwrap

from pathlib import Path
from rich.panel import Panel
from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.paths import (
    nbs_dir, settings_dict, resolve_selector, iter_notebooks,
    read_nb, abs_module_for_nb,
)
from nbdev_mcp.utils.nb import join_source, find_default_exp
from nbdev_mcp.utils.rich import render_table, render_panel

## Dependency Analysis

In [ ]:
#| export
def analyze_dependency_order(
    project: Optional[str] = None,
    enforce: bool = False
) -> Dict[str, Any]:
    """Analyze notebook dependency graph to suggest numbering order.
    
    Ranks notebooks by their internal import dependencies to determine
    the optimal ordering. Optionally enforces the suggested numbering
    by renaming notebooks.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    enforce : bool
        If True, rename notebooks to match suggested order.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'suggestions' list and 'applied' count if enforce=True.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    nbs_root = nbs_dir(p)
    
    # Build dependency graph
    nodes: Dict[str, 'Path'] = {}
    imports: Dict[str, Set[str]] = {}
    
    for nb in iter_notebooks(p):
        if not str(nb).startswith(str(nbs_root)) or nb.name.startswith('00__init__'):
            continue
        
        mod = find_default_exp(read_nb(nb)) or abs_module_for_nb(p, nb)[1]
        fq = f'{lib}.{mod}'
        nodes[fq] = nb
        imports[fq] = set()
        
        data = read_nb(nb)
        for cell in data.get('cells', []):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            # Find from ... import statements
            for m in re.finditer(r'^\s*from\s+([\w\.]+)\s+import\s+', src, flags=re.MULTILINE):
                tmod = m.group(1)
                if tmod.startswith(f'{lib}.'):
                    imports[fq].add(tmod)
            
            # Find import statements
            for m in re.finditer(r'^\s*import\s+([\w\.]+)', src, flags=re.MULTILINE):
                tmod = m.group(1)
                if tmod.startswith(f'{lib}.'):
                    imports[fq].add(tmod)
    
    # Calculate ranks (topological sort with levels)
    rank: Dict[str, int] = {k: 1 for k in nodes}
    changed = True
    iters = 0
    while changed and iters < 1000:
        changed = False
        iters += 1
        for a, deps in imports.items():
            for b in deps:
                if b in rank and rank[a] <= rank[b]:
                    rank[a] = rank[b] + 1
                    changed = True
    
    # Generate suggestions
    suggestions: List[Dict[str, Any]] = []
    for fq, nb in nodes.items():
        rel = nb.relative_to(nbs_root)
        cur_name = rel.name
        base_name = re.sub(r'^\d+_', '', cur_name)
        rn = f'{rank[fq]:02d}_{base_name}'
        if rn != cur_name:
            suggestions.append({
                'notebook': str(nb.relative_to(p)),
                'suggested': str(rel.with_name(rn)),
                'reason': f'rank {rank[fq]} from dependency graph'
            })
    
    # Apply if enforcing
    applied = 0
    if enforce:
        for s in suggestions:
            src = p / s['notebook']
            dst = p / s['suggested']
            dst.parent.mkdir(parents=True, exist_ok=True)
            os.replace(src, dst)
            applied += 1
    
    rows = [[s['notebook'], s['suggested'], s['reason']] for s in suggestions[:200]]
    pretty = render_table('numbering suggestions', ['notebook', 'suggested', 'why'], rows)
    if enforce:
        pretty += '\n' + render_panel('enforced', f'Applied renames: {applied}')
    
    return {'ok': True, 'suggestions': suggestions, 'applied': applied, 'pretty': pretty}

In [ ]:
#| export
def dependency_tree(
    project: Optional[str] = None,
    scope: str = 'internal',
    include_unused: Optional[str] = None,
    write_qmd: Optional[str] = None
) -> Dict[str, Any]:
    """Summarize internal and external module dependencies.
    
    Analyzes import statements across all notebooks to build a
    dependency graph. Can generate Mermaid and DOT diagrams.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    scope : str
        'internal', 'external', or 'both' to control which deps to show.
    include_unused : str, optional
        'internal', 'external', or 'both' to include unused imports.
    write_qmd : str, optional
        Path to write a Quarto document with diagrams.
    
    Returns
    -------
    Dict[str, Any]
        Result with nodes, edges, mermaid/dot diagrams, and optional written file.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    
    internal_nodes: Set[str] = set()
    external_nodes: Set[str] = set()
    edges: Set[Tuple[str, str]] = set()
    in_deg: Dict[str, int] = {}
    unused_external: List[Dict[str, Any]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        mod = find_default_exp(data) or abs_module_for_nb(p, nb)[1]
        me = f'{lib}.{mod}'
        internal_nodes.add(me)
        in_deg.setdefault(me, 0)
        
        for ci, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            try:
                tree = ast.parse(src)
            except SyntaxError:
                tree = None
            
            imported_aliases: Set[str] = set()
            used_names: Set[str] = set()
            
            if tree is not None:
                for node in ast.walk(tree):
                    if isinstance(node, ast.Import):
                        for n in node.names:
                            alias = n.asname or n.name.split('.')[0]
                            imported_aliases.add(alias)
                            if not n.name.startswith(f'{lib}.'):
                                external_nodes.add(n.name)
                                if scope in ('external', 'both'):
                                    edges.add((me, n.name))
                    elif isinstance(node, ast.ImportFrom):
                        modname = node.module or ''
                        for n in node.names:
                            alias = n.asname or n.name
                            imported_aliases.add(alias)
                        if modname:
                            if modname.startswith(f'{lib}.'):
                                tgt = modname
                                if scope in ('internal', 'both'):
                                    edges.add((me, tgt))
                                    in_deg[tgt] = in_deg.get(tgt, 0) + 1
                            else:
                                external_nodes.add(modname)
                                if scope in ('external', 'both'):
                                    edges.add((me, modname))
                
                for node in ast.walk(tree):
                    if isinstance(node, ast.Name):
                        used_names.add(node.id)
            
            for alias in imported_aliases:
                if alias not in used_names:
                    unused_external.append({
                        'notebook': str(nb.relative_to(p)),
                        'cell': ci,
                        'alias': alias
                    })
    
    unreferenced_internal = [n for n in sorted(internal_nodes) if in_deg.get(n, 0) == 0]
    
    # Generate Mermaid diagram
    mermaid = ['graph LR\n']
    for a, b in sorted(edges):
        mermaid.append(f"  {a.replace('.', '_')}[{a}] --> {b.replace('.', '_')}[{b}]\n")
    
    # Generate DOT diagram
    dot = ['digraph G {\n', '  rankdir=LR;\n']
    for a, b in sorted(edges):
        dot.append(f'  "{a}" -> "{b}";\n')
    dot.append('}\n')
    
    # Write QMD if requested
    written = None
    if write_qmd:
        qmd = p / write_qmd
        qmd.parent.mkdir(parents=True, exist_ok=True)
        mermaid_block = ''.join(mermaid)
        dot_block = ''.join(dot)
        qmd_content = f"""---
title: Dependency Tree for {lib}
format: html
---

> This diagram shows module import dependencies.

## Mermaid
```{{{{mermaid}}}}
{mermaid_block}
```

## Graphviz DOT
```{{{{dot}}}}
{dot_block}
```
"""
        qmd.write_text(qmd_content, encoding='utf-8')
        written = str(qmd.relative_to(p))
    
    pretty = render_panel('dependency tree', 'Dependency tree generated (internal/external).')
    
    return {
        'ok': True,
        'scope': scope,
        'nodes_internal': sorted(internal_nodes) if scope in ('internal', 'both') else [],
        'nodes_external': sorted(external_nodes) if scope in ('external', 'both') else [],
        'edges': sorted(list(edges)),
        'unreferenced_internal': unreferenced_internal if include_unused in ('internal', 'both') else [],
        'unused_external_aliases': unused_external if include_unused in ('external', 'both') else [],
        'mermaid': ''.join(mermaid),
        'dot': ''.join(dot),
        'written': written,
        'pretty': pretty
    }

## MCP Registration

In [ ]:
#| export
def add_analysis_tools(mcp: FastMCP) -> None:
    """Attach dependency analysis tools to the MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    mcp.add_tool(analyze_dependency_order)
    mcp.add_tool(dependency_tree)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()